# Predicting Commuting Time in South Africa

## 1. Data Loading and Initial Exploration

In [17]:
# Importing libraries for data manipulation, ZIP file handling, and file paths
import pandas as pd
import zipfile
from pathlib import Path

In [18]:
# Defining the path to the compressed survey dataset
dataset_path = Path("../data/nhts-2020-v1.zip")

# Opening the ZIP file and loading the Stata person-level dataset
with zipfile.ZipFile(dataset_path) as z:
    with z.open("nhts-2020-v1/nhts-2020-v1-stata/nhts-2020-person.dta") as f:
        df = pd.read_stata(f, convert_categoricals=False)

# Checking the number of rows and columns in the dataset
print(df.shape)

(145385, 274)


In [19]:
#Selecting eight independent variables

features= [
    "Age",
    "Sex",
    "Q43SectOfWork",
    "Q48bDMunic",
    "Q49Days",
    "Q420OffiTime",
    "Q413EmployCash",
    "Mainmode_work"
]

#Defining what we want to predict

target = "Tot_time_travel_work"

#Selecting the features and target from the original dataset

commute_df = df[features + [target]].copy()

#Checking the dimensions
print("Dataset shape:", commute_df.shape)

#Displaying the first five observations

commute_df.head()

Dataset shape: (145385, 9)


,Age,Sex,Q43SectOfWork,Q48bDMunic,Q49Days,Q420OffiTime,Q413EmployCash,Mainmode_work,Tot_time_travel_work
0,66,2,8,99,9,9999,9,8,999
1,39,1,1,12,5,0730,2,6,20
2,7,1,9,99,9,9999,9,8,999
3,25,1,1,12,5,0700,2,6,20
4,23,2,8,99,9,9999,9,8,999


In [20]:
# Examining the distribution of commuting times

print(commute_df["Tot_time_travel_work"].describe())

#Count unspecified commuting times

print(
    "Unspecified records:",
    (commute_df["Tot_time_travel_work"] == 999). sum()
)

count    145385.000000
mean        821.913340
std         371.267026
min           1.000000
25%         999.000000
50%         999.000000
75%         999.000000
max         999.000000
Name: Tot_time_travel_work, dtype: float64
Unspecified records: 118404


In [21]:
# Removing invalid target values

commute_df = commute_df[ commute_df["Tot_time_travel_work"].between(1, 365)].copy()
print("Remaining records:", len(commute_df))
print(commute_df["Tot_time_travel_work"].describe())

Remaining records: 26981
count    26981.000000
mean        44.782439
std         34.332372
min          1.000000
25%         20.000000
50%         35.000000
75%         60.000000
max        365.000000
Name: Tot_time_travel_work, dtype: float64


In [22]:
# Checking missing values in each of the selected column

print(commute_df.isnull().sum())

#Examining the number of unique values in each feature

print(commute_df[features].nunique())

Age                     0
Sex                     0
Q43SectOfWork           0
Q48bDMunic              0
Q49Days                 0
Q420OffiTime            0
Q413EmployCash          0
Mainmode_work           0
Tot_time_travel_work    0
dtype: int64
Age                72
Sex                 2
Q43SectOfWork       5
Q48bDMunic         53
Q49Days             8
Q420OffiTime      135
Q413EmployCash      2
Mainmode_work       8
dtype: int64


In [23]:
# Identifying records with valid values across the seven selected features
valid = (
    commute_df["Age"].between(15,120)
    & commute_df["Sex"].isin([1,2])
    & commute_df["Q43SectOfWork"].isin ([1,2,3])
    & commute_df["Q48bDMunic"].between (11,70)
    & commute_df["Q49Days"].between (1,7)
    & commute_df["Q413EmployCash"].isin([1,2])
    & commute_df["Mainmode_work"].between(1,7)

)
print("Records passing seven feature checks:", valid.sum())

Records passing seven feature checks: 24492


In [24]:
# Converting work start "times to strings
time= commute_df["Q420OffiTime"].astype(str).str.zfill(4)

# Checking whether hours and minutes are valid

valid_time= (
    time.str.fullmatch(r"\d{4}")
    & (pd.to_numeric(time.str[:2], errors="coerce") < 24)
    & (pd.to_numeric(time.str[:2], errors="coerce") < 60)
)

# Combining all the eight feature checks

valid= valid & valid_time

print("Records passing all eight checks:", valid.sum())

Records passing all eight checks: 24492


In [25]:
# Keeping only records that passed all eight checks
commute_df = commute_df[valid].copy()

# Resetting the row numbering
commute_df = commute_df.reset_index(drop=True)

# Checking the cleaned dataset
print("Final dataset shape:", commute_df.shape)

commute_df.head()

Final dataset shape: (24492, 9)


,Age,Sex,Q43SectOfWork,Q48bDMunic,Q49Days,Q420OffiTime,Q413EmployCash,Mainmode_work,Tot_time_travel_work
0,39,1,1,12,5,0730,2,6,20
1,25,1,1,12,5,0700,2,6,20
2,62,2,1,12,5,0730,2,6,10
3,37,1,1,12,5,0700,2,4,25
4,28,1,1,12,5,0700,2,4,25
